# Jev API로 임상·Holter 데이터 SCD 분류 실험

Jev의 **noul(참/거짓) 점수**로 4년 이내 SCD 발생 여부를 예측합니다. Jev를 학습시키지 않는 **zero-shot** 실험이며, 입력 구성을 두 가지 버전 중 `INPUT_VERSION` 변수로 고릅니다.

| `INPUT_VERSION` | 입력 |
|---|---|
| `"clinical_holter"` | 임상 23개 변수(기존 ML v4와 동일) + Holter 요약 특징 |
| `"holter_only"` | Holter 요약 특징만 |

- 입력: `clinical_clean.csv`, 기존 환자 분할 `final_split.csv`, Holter 파형 `datas/Holter/windows`
- Holter 파형은 원본 신호 대신 RR 간격 기반 **요약 특징**(심박수, HRV, 감속능 추정치, 품질 지표)으로 바꿔 입력합니다. 계산은 기존 `monorouter_holter` 코드와 캐시를 재사용합니다.
- 기본 평가: **test 분할에서 양성 10명 + 음성 10명**(`SAMPLE_PER_CLASS`)만 사용합니다. 음성은 시드 고정 무작위 추출입니다. `EVALUATION_SPLITS`와 `SAMPLE_PER_CLASS`로 바꿀 수 있습니다.
- 출력: 환자별 점수 CSV, 성능 CSV, 점수 분포도 (실행 폴더 이름에 버전이 붙습니다)

**연구용:** 점수의 임상적 확률 보정과 의료적 유용성은 검증되지 않았습니다. test 결과를 보고 프롬프트나 기준을 고쳐 다시 평가하면 독립 성능이 아니게 되므로 주의하세요. [Jev AI API 문서](https://jev-ai.pro/docs)

## 1. 라이브러리와 저장소 위치

`Global` 환경의 pandas, requests, python-dotenv, scikit-learn, matplotlib를 사용합니다. 저장소 내부 어느 작업 폴더에서 실행해도 루트를 찾습니다.

In [ ]:
import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import yaml
from dotenv import load_dotenv
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score

In [ ]:
def find_repository(start):
    for folder in [start, *start.parents]:
        if (folder / "Members/sang/model/ML/feature_versions.py").is_file():
            return folder
    raise FileNotFoundError("sixletters-poc 저장소 안에서 실행해 주세요.")


REPO = find_repository(Path.cwd().resolve())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from Members.sang.model.ML.feature_versions import MODEL_4  # noqa: E402

print("저장소:", REPO.name)
print("임상 입력 변수:", len(MODEL_4), "개")

## 2. 실행 설정

`Members/sang/notebooks/.env`에 `JEV_API_KEY=발급받은키`를 입력하세요. 기존 시스템 환경 변수도 사용할 수 있습니다. 파일 내용과 키를 출력하지 않습니다.

**`INPUT_VERSION`으로 실험 버전을 선택합니다.** 두 버전 모두 같은 환자 집합을 사용하므로 결과를 짝지어 비교할 수 있습니다.

먼저 `RUN_API = False`로 데이터와 설정을 확인하세요. 실제 호출할 때 `RUN_API = True`로 바꾸면 **선택한 버전의 입력(임상 수치와 Holter 요약 특징)이 외부 Jev AI API에 전송**되며 호출 비용이 발생할 수 있습니다. 환자 ID와 실제 정답은 요청에 포함하지 않지만 임상·Holter 수치 자체는 민감 정보입니다.

In [ ]:
DATA_DIR = REPO / "Members/sang/datas/clincal"
CLINICAL_CSV = DATA_DIR / "clinical_clean.csv"
SPLIT_CSV = DATA_DIR / "final_split.csv"
ENV_PATH = REPO / "Members/sang/notebooks/.env"
RESULTS_DIR = REPO / "Members/sang/results/jev_clinical"

# 실험 버전 선택: "clinical_holter"(임상+Holter) 또는 "holter_only"(Holter만)
INPUT_VERSION = "clinical_holter"
INPUT_VERSIONS = {
    "clinical_holter": "임상 23개 변수 + Holter 요약 특징",
    "holter_only": "Holter 요약 특징만",
}
if INPUT_VERSION not in INPUT_VERSIONS:
    raise ValueError(f"INPUT_VERSION은 {list(INPUT_VERSIONS)} 중 하나여야 합니다.")
USE_CLINICAL = INPUT_VERSION == "clinical_holter"

BASIC_YAML = REPO / "Members/sang/model/basic.yaml"
HOLTER_YAML = REPO / "Members/sang/notebooks/monorouter_holter.yaml"
HOLTER_REFERENCE_CSVS = [  # 환자별 Holter 리듬 코드(로컬 계산용, API로 보내지 않음)
    REPO / "Members/sang/datas/Holter/physionet_reference/rhythm_hrv_dev.csv",
    REPO / "Members/sang/datas/Holter/physionet_reference/rhythm_hrv_test.csv",
]

MODEL = "jev-latest"
API_URL = "https://jev-ai.pro/api/v1/systemone"
EVALUATION_SPLITS = ["test"]  # 예: ["train", "validation"]
# 클래스별로 이 인원만 뽑아 양성·음성 수를 맞춥니다. None이면 선택한 분할 전체를 사용합니다.
SAMPLE_PER_CLASS = None
SAMPLE_SEED = 42  # 같은 값이면 두 INPUT_VERSION이 항상 같은 환자를 씁니다.
THRESHOLD = 0.15
RUN_API = True

load_dotenv(ENV_PATH, override=False)
print("실험 버전:", INPUT_VERSION, "-", INPUT_VERSIONS[INPUT_VERSION])
print("API 호출:", "켜짐" if RUN_API else "꺼짐")
print("평가 분할:", EVALUATION_SPLITS)
print("클래스별 표본:", SAMPLE_PER_CLASS if SAMPLE_PER_CLASS else "전체", "| 시드:", SAMPLE_SEED)

## 3. 데이터 불러오기와 검증

임상 CSV와 분할 CSV를 환자 ID로 일대일 결합합니다. 중복 ID, 누락된 변수·환자, 양쪽의 정답 불일치는 오류로 처리합니다. 결과를 보기 전 환자별 행이나 실제 임상 수치는 출력하지 않습니다.

In [ ]:
clinical = pd.read_csv(CLINICAL_CSV, dtype={"Patient ID": str})
split = pd.read_csv(SPLIT_CSV, dtype={"pid": str})
clinical.columns = clinical.columns.str.strip()
split.columns = split.columns.str.strip()
split["split"] = split["split"].replace({"val": "validation"})

if clinical.columns.duplicated().any() or split.columns.duplicated().any():
    raise ValueError("중복 열 이름이 있습니다.")
if clinical["Patient ID"].duplicated().any() or split["pid"].duplicated().any():
    raise ValueError("중복 환자 ID가 있습니다.")
if not set(MODEL_4).issubset(clinical.columns):
    raise ValueError("임상 v4 입력 변수가 누락됐습니다.")
if not split["split"].isin(["train", "validation", "test"]).all():
    raise ValueError("알 수 없는 분할 값이 있습니다.")

In [ ]:
cohort = split.merge(
    clinical,
    left_on="pid",
    right_on="Patient ID",
    how="left",
    validate="one_to_one",
    suffixes=("_split", "_clinical"),
    indicator=True,
)
if (cohort["_merge"] != "both").any():
    raise ValueError("임상 CSV에서 찾지 못한 환자가 있습니다.")
if not cohort["SCD_split"].isin([0, 1]).all():
    raise ValueError("SCD 정답은 0 또는 1이어야 합니다.")
if not (cohort["SCD_split"] == cohort["SCD_clinical"]).all():
    raise ValueError("두 CSV의 SCD 정답이 다릅니다.")

cohort = cohort[cohort["split"].isin(EVALUATION_SPLITS)].copy()
cohort[MODEL_4] = cohort[MODEL_4].apply(pd.to_numeric, errors="coerce")
if not np.isfinite(cohort[MODEL_4].to_numpy(dtype=float)).all():
    raise ValueError("입력 변수에 결측값 또는 무한대가 있습니다.")
if cohort.empty:
    raise ValueError("평가 대상 환자가 없습니다.")

if SAMPLE_PER_CLASS is not None:
    groups = []
    for label in (0, 1):
        members = cohort[cohort["SCD_split"] == label]
        if len(members) < SAMPLE_PER_CLASS:
            raise ValueError(
                f"SCD={label} 환자가 {SAMPLE_PER_CLASS}명보다 적습니다: {len(members)}명"
            )
        groups.append(members.sample(n=SAMPLE_PER_CLASS, random_state=SAMPLE_SEED))
    cohort = pd.concat(groups).sort_values("pid").reset_index(drop=True)

In [ ]:
print("평가 환자:", len(cohort), "명")
print("실제 양성:", int(cohort["SCD_split"].sum()), "명")
print("실제 음성:", int((cohort["SCD_split"] == 0).sum()), "명")
print("분할별 인원:", cohort["split"].value_counts().to_dict())

## 3-1. Holter 특징 준비

`datas/Holter/windows`의 원본 파형에서 기존 `monorouter_holter` 코드로 RR 간격 기반 요약 특징을 만듭니다. 환자별 캐시가 있으면 재사용하고, 없는 환자만 새로 계산하므로 첫 계산은 오래 걸릴 수 있습니다. 이 단계는 로컬에서만 실행되며 외부 전송은 없습니다.

- 대상은 증강 복사본(`P0001-1` 등)을 제외한, 평가 코호트(표본 추출 후)에 해당하는 원본 환자입니다. 리듬 코드는 개발용(`rhythm_hrv_dev.csv`)과 test용(`rhythm_hrv_test.csv`) 참고 파일에서 읽습니다.
- 임상 코호트와 환자 ID·분할·정답이 하나라도 다르거나 특징 계산에 실패한 환자가 있으면 오류로 중단합니다. 두 버전이 같은 환자 집합을 쓰도록 하기 위해서입니다.
- 특징 계산법은 연구용이며 박동 종류 검증은 끝나지 않았습니다. 자세한 한계는 `docs/holter_llm_feature_method.md`를 참고하세요.

In [ ]:
import csv
import re

from Members.sang.experiments.monorouter_holter import clinical_features as holter_cf  # noqa: E402

COHORT_IDS = set(cohort["pid"])  # 표본으로 뽑힌 환자만 Holter 특징을 준비합니다.


def select_holter_records(split_path, wave_dir, reference_path, sample_size, seed):
    """평가 코호트의 원본 환자만 고릅니다. 증강 복사본(-1, -2)은 제외합니다."""
    reference = {}
    for path in HOLTER_REFERENCE_CSVS:
        with path.open(encoding="utf-8-sig", newline="") as stream:
            for row in csv.DictReader(stream):
                pid = row["patient_id"].strip()
                if pid in reference:
                    raise ValueError("리듬 참고 정보에 중복 ID가 있습니다.")
                reference[pid] = row
    records = []
    for pid, group, label in holter_cf.read_patient_split(Path(split_path)):
        if group not in EVALUATION_SPLITS or pid not in COHORT_IDS or re.search(r"-\d+$", pid):
            continue
        wave_path = Path(wave_dir) / f"{pid}.npy"
        if not wave_path.is_file():
            raise FileNotFoundError(f"원본 파형이 없습니다: {pid}")
        if pid not in reference:
            raise ValueError(f"리듬 참고 정보가 없습니다: {pid}")
        records.append(
            {
                "patient_id": pid,
                "split": group,
                "label": int(label),
                "wave_path": wave_path,
                "reference": reference[pid],
            }
        )
    if not records:
        raise ValueError("선택한 분할의 Holter 원본 기록이 없습니다.")
    return records


holter_cf.select_development = select_holter_records  # prepare_development가 이 선택기를 씁니다.

In [ ]:
import contextlib
import io

from Members.sang.model.config import load_basic_config  # noqa: E402

HOLTER_SETTING_KEYS = (
    "chunk_seconds",
    "context_seconds",
    "min_rr_ms",
    "max_rr_ms",
    "relative_limit",
    "sdann_min_coverage",
    "hourly_min_coverage",
)
holter_config = yaml.safe_load(HOLTER_YAML.read_text(encoding="utf-8"))
holter_settings = {key: holter_config["clinical"][key] for key in HOLTER_SETTING_KEYS}
basic_config = load_basic_config(BASIC_YAML)

with contextlib.redirect_stdout(io.StringIO()):  # 환자별 진행 로그는 숨깁니다.
    holter_records, holter_prepared = holter_cf.prepare_development(
        REPO,
        basic_config,
        BASIC_YAML,
        HOLTER_REFERENCE_CSVS[0],  # 선택기가 두 참고 파일을 모두 읽으므로 형식상 전달
        None,
        holter_config["experiment"]["sample_seed"],
        holter_settings,
    )

holter_failed = [pid for pid, _, features, _ in holter_prepared if features is None]
if holter_failed:
    raise RuntimeError(f"Holter 특징 계산 실패: {len(holter_failed)}명")
holter_features = {pid: features for pid, _, features, _ in holter_prepared}
holter_meta = pd.DataFrame(holter_records).set_index("patient_id")

expected = cohort.set_index("pid")
if set(holter_features) != set(expected.index):
    raise ValueError("Holter 환자 집합이 임상 코호트와 다릅니다.")
if not (holter_meta.loc[expected.index, "label"] == expected["SCD_split"]).all():
    raise ValueError("Holter와 임상 CSV의 SCD 정답이 다릅니다.")
if not (holter_meta.loc[expected.index, "split"] == expected["split"]).all():
    raise ValueError("Holter와 임상 CSV의 분할이 다릅니다.")

print("Holter 특징 준비:", len(holter_features), "명")

## 4. Jev 요청 만들기

`state`에는 선택한 버전의 입력만 담습니다. `clinical_holter`는 `clinical`(임상 23개 변수)과 `holter`(요약 특징) 두 묶음을, `holter_only`는 `holter`만 보냅니다. `questions`는 4년 이내 SCD 발생을 묻는 `noul` 한 개입니다. 요청에 SCD 정답, 환자 ID, 분할 이름은 넣지 않습니다. 의료진 설명이나 파형 판독은 요청하지 않습니다.

In [ ]:
INSTRUCTIONS = {
    "clinical_holter": (
        "Research only. Based on the provided clinical variables and the "
        "Holter-derived RR/HRV summary features, will the patient experience "
        "sudden cardiac death within four years after the Holter recording? "
        "Holter values are experimental estimates; null means unavailable. "
        "Treat missing data as unknown."
    ),
    "holter_only": (
        "Research only. Based only on the provided Holter-derived RR/HRV summary "
        "features, will the patient experience sudden cardiac death within four years "
        "after the Holter recording? Holter values are experimental estimates; "
        "null means unavailable. Treat missing data as unknown."
    ),
}


def make_state(patient):
    state = {}
    if USE_CLINICAL:
        state["clinical"] = {name: float(patient[name]) for name in MODEL_4}
    state["holter"] = holter_features[patient["pid"]]
    return state


def make_payload(patient):
    return {
        "model": MODEL,
        "state": json.dumps(make_state(patient), ensure_ascii=True, allow_nan=False),
        "questions": {
            "scd_within_4_years": {
                "type": "noul",
                "instructions": INSTRUCTIONS[INPUT_VERSION],
            }
        },
    }


example_state = make_state(cohort.iloc[0])
print("실험 버전:", INPUT_VERSION)
print("state 최상위 항목:", list(example_state))
if USE_CLINICAL:
    print("임상 변수 개수:", len(example_state["clinical"]))
print("정답과 환자 ID는 Jev 요청에 포함하지 않습니다.")

## 5. API 호출과 응답 확인

공식 응답의 `answers.scd_within_4_years.noul`을 양성 점수로 읽습니다. 숫자가 아니거나 0~1 범위를 벗어나면 실패로 기록합니다. 429·일시적인 서버 오류만 제한적으로 재시도하고 인증·크레딧 오류는 전체 실행을 중단합니다. 오류 응답 본문이나 API 키는 출력하지 않습니다.

In [ ]:
def read_probability(response_data):
    answer = response_data["answers"]["scd_within_4_years"]
    score = answer["noul"]
    if answer.get("type") != "noul" or isinstance(score, bool):
        raise ValueError("noul 응답 형식이 잘못됐습니다.")
    if not isinstance(score, (int, float)) or not np.isfinite(score):
        raise ValueError("점수가 유효한 숫자가 아닙니다.")
    if not 0 <= score <= 1:
        raise ValueError("점수가 0~1 범위를 벗어났습니다.")
    return float(score)

In [ ]:
def request_probability(session, patient, api_key, max_attempts=3):
    headers = {"Authorization": f"Bearer {api_key}"}
    payload = make_payload(patient)
    for attempt in range(max_attempts):
        try:
            response = session.post(API_URL, headers=headers, json=payload, timeout=60)
        except requests.RequestException:
            if attempt + 1 == max_attempts:
                raise RuntimeError("API 연결 실패") from None
        else:
            if response.status_code in (401, 402, 403):
                raise PermissionError(f"API 인증 또는 이용 권한 오류: {response.status_code}")
            if response.status_code == 200:
                return read_probability(response.json())
            if response.status_code not in (429, 500, 502, 503, 504):
                raise RuntimeError(f"API 요청 실패: HTTP {response.status_code}")
            if attempt + 1 == max_attempts:
                raise RuntimeError(f"API 재시도 실패: HTTP {response.status_code}")
        time.sleep(2**attempt)
    raise RuntimeError("API 요청 실패")

## 6. 환자별 추론

한 환자당 한 번 요청합니다. 각 결과는 즉시 CSV에 저장하므로 중간에 중단돼도 완료된 행을 확인할 수 있습니다. 각 실행은 별도 폴더를 사용합니다. API 호출을 끄면 네트워크 요청이나 결과 파일 생성은 하지 않습니다.

In [ ]:
def create_result_folder():
    run_name = datetime.now().strftime("%Y%m%d_%H%M%S_%f") + f"_{INPUT_VERSION}"
    folder = RESULTS_DIR / run_name
    folder.mkdir(mode=0o700, parents=True, exist_ok=False)
    return folder


def append_prediction(path, result):
    file_exists = path.exists()
    pd.DataFrame([result]).to_csv(path, mode="a", index=False, header=not file_exists)
    path.chmod(0o600)

In [ ]:
results = []
run_dir = None

if RUN_API:
    api_key = os.getenv("JEV_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError(".env 또는 환경 변수에 JEV_API_KEY를 설정하세요.")
    run_dir = create_result_folder()
    with requests.Session() as session:
        for index, (_, patient) in enumerate(cohort.iterrows(), start=1):
            result = {
                "patient_id": patient["pid"],
                "split": patient["split"],
                "y_true": int(patient["SCD_split"]),
                "p_positive": None,
                "status": "error",
            }
            try:
                result["p_positive"] = request_probability(session, patient, api_key)
                result["status"] = "ok"
            except PermissionError:
                raise
            except (RuntimeError, ValueError, KeyError, TypeError):
                pass
            results.append(result)
            append_prediction(run_dir / "patient_predictions.csv", result)
            if index % 25 == 0 or index == len(cohort):
                print("완료:", index, "/", len(cohort))
else:
    print("API 호출을 건너뛰었습니다. 실제 실행은 RUN_API = True로 변경하세요.")

## 7. 환자 단위 성능과 양성 탐지

정상 응답을 받은 환자만 집계하고 전체 대상 대비 성공 수를 함께 표시합니다. AUROC·AP는 점수 순위를 평가합니다. 50% 기준은 실험 비교용이며 의료 알림 기준이 아닙니다. 실패가 있으면 전체 대상 환자의 성능으로 해석하면 안 됩니다.

In [ ]:
if RUN_API:
    predictions = pd.DataFrame(results)
    valid = predictions[predictions["status"] == "ok"].copy()
    print("성공:", len(valid), "/", len(cohort), "명")
    print("실패:", len(predictions) - len(valid), "명")

    if len(valid) == 0 or valid["y_true"].nunique() != 2:
        raise ValueError("AUROC/AP 계산에 필요한 양성과 음성이 모두 있어야 합니다.")

    y_true = valid["y_true"].astype(int)
    scores = valid["p_positive"].astype(float)
    tn, fp, fn, tp = confusion_matrix(y_true, scores >= THRESHOLD, labels=[0, 1]).ravel()
    metrics = {
        "model": MODEL,
        "input": INPUT_VERSION,
        "threshold": THRESHOLD,
        "expected": len(cohort),
        "succeeded": len(valid),
        "auroc": roc_auc_score(y_true, scores),
        "average_precision": average_precision_score(y_true, scores),
        "positive_detected": int(tp),
        "positive_missed": int(fn),
        "false_positive": int(fp),
        "true_negative": int(tn),
    }
    pd.DataFrame([metrics]).to_csv(run_dir / "metrics.csv", index=False)
    (run_dir / "metrics.csv").chmod(0o600)
    display(pd.DataFrame([metrics]))

## 8. 실제 양성·음성의 점수 분포

양성과 음성의 수가 다르므로 각 집단 안에서의 비율로 표시합니다. 점선은 실험용 50% 기준입니다.

In [ ]:
if RUN_API:
    fig, ax = plt.subplots(figsize=(9, 5))
    bins = np.linspace(0, 1, 21)
    for label, color, title in [
        (0, "steelblue", "Actual negative"),
        (1, "tomato", "Actual positive"),
    ]:
        values = valid.loc[valid["y_true"] == label, "p_positive"].astype(float)
        ax.hist(
            values,
            bins=bins,
            weights=np.ones(len(values)) / len(values),
            alpha=0.55,
            color=color,
            label=f"{title} (n={len(values)})",
        )
    ax.axvline(THRESHOLD, color="black", linestyle="--", label="Threshold")
    ax.set(
        xlabel="Jev score",
        ylabel="Fraction of each class",
        title=f"Jev score distribution ({INPUT_VERSION})",
    )
    ax.legend()
    fig.tight_layout()
    fig.savefig(run_dir / "score_distribution.png", dpi=160)
    (run_dir / "score_distribution.png").chmod(0o600)
    plt.show()
    print("결과 폴더:", run_dir)

## 해석할 때

- 두 버전(`clinical_holter`, `holter_only`)은 같은 환자 집합·같은 모델·같은 임계값을 사용하므로 `INPUT_VERSION`만 바꿔 각각 실행한 뒤 `metrics.csv`를 비교할 수 있습니다. 차이가 Holter 정보의 기여라고 단정하려면 표본 수(양성 10명, 음성 10명)에 따른 불확실성을 함께 봐야 합니다. 음성은 79명 중 일부만 뽑은 것이라 시드가 달라지면 결과가 달라질 수 있습니다.
- Holter 원본 파형이나 30초 구간을 Jev에 직접 입력하지 않고, RR 기반 요약 특징만 보냅니다. 이 특징은 박동 종류가 검증되지 않은 연구용 추정치입니다.
- 기존 개발 데이터 실험과 비교할 수 있지만 모델·입력·프롬프트가 다릅니다. 결과 차이가 Jev 자체의 우수성을 증명하지는 않습니다.
- 측정 시점에 실제로 알 수 있었던 변수인지 확인해야 하며, test와 외부 데이터 검증 및 확률 보정이 필요합니다.
- 환자별 결과 CSV는 민감 정보를 포함하므로 Git에 올리지 않습니다. API 실행이 중단되면 이미 저장된 CSV를 확인하고 별도 새 실행을 시작하세요.